# TouchTrace — Touch Model Training (Colab)

Train the Phase 1 touch LSTM + MDN on CSD4CA data and export ONNX weights.

**Before you start:** Runtime → Change runtime type → **GPU** (T4).

Repo: [github.com/ginwzy/TouchTrace](https://github.com/ginwzy/TouchTrace)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repository

Training data `touch_data.jsonl.gz` (~13 MB) is already in the repo.

In [ ]:
from pathlib import Path

REPO = "TouchTrace"
REPO_URL = "https://github.com/ginwzy/TouchTrace.git"
ROOT = Path("/content") / REPO
TOUCH_DIR = ROOT / "train" / "touch"

if not ROOT.exists():
    !git clone {REPO_URL}
else:
    !git -C {ROOT} pull --ff-only

assert TOUCH_DIR.is_dir(), f"Missing directory: {TOUCH_DIR}"
%cd {TOUCH_DIR}

data = TOUCH_DIR / "touch_data.jsonl.gz"
assert data.is_file(), f"Missing {data} (cwd={Path.cwd()})"
print(f"Data: {data} ({data.stat().st_size / 1e6:.1f} MB)")
!ls -lh touch_data.jsonl.gz

## 3. Install dependencies

In [ ]:
!pip install -q tensorflow tensorflow-probability tf-keras tf2onnx onnxruntime matplotlib pytest

## 4. Verify TensorFlow sees the GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", gpus)

if not gpus:
    print("\n⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU, then rerun from the top.")
else:
    print("\n✓ GPU ready.")

## 5. Train

Adjust the options below, then run the cell.

Training defaults (from `config_touch.py`): subsample 3px, weighted loss, input noise, 250 max epochs, batch 256, prepad on GPU.

| Option | Default | Description |
|--------|---------|-------------|
| `EPOCHS` | `None` | Max epochs; `None` uses config (250) |
| `LITE` | `False` | Use 2×64 LSTM instead of 2×128 |
| `SEQUENCE` | `False` | Mac Metal fallback only; Colab/CUDA use prepad |

In [ ]:
EPOCHS = None  # None → config default (250)
LITE = False
SEQUENCE = False

cmd = ["python", "train_touch.py"]
if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LITE:
    cmd.append("--lite")
if SEQUENCE:
    cmd.append("--sequence")

print("Running:", " ".join(cmd))
!{" ".join(cmd)}

## 6. (Optional) Run unit tests

In [ ]:
!python -m pytest -q

## 7. Export ONNX

In [ ]:
export_cmd = "python convert_touch.py"
if LITE:
    export_cmd += " --lite"

!{export_cmd}
!ls -lh model.h5 model_best.h5 model_last.h5 *.onnx 2>/dev/null || ls -lh model*.h5

## 8. Evaluate generation by swipe angle

Report raw / no-backtrack / guided, split into H/D/V. Target: diagonal and horizontal reach close to vertical.

In [ ]:
!python eval_generate.py --model touch.onnx --limit 200
!python preview_swipes.py --model touch.onnx --out-dir /content/plots --from-data --raw
!python preview_swipes.py --model touch.onnx --out-dir /content/plots --from-data --noback
!python preview_swipes.py --model touch.onnx --out-dir /content/plots --from-data

## 9. Download weights to your computer

In [ ]:
from google.colab import files
from pathlib import Path

downloads = ["model_best.h5", "model.h5"]
onnx = "touch_lite.onnx" if LITE else "touch.onnx"
if Path(onnx).exists():
    downloads.append(onnx)

for name in downloads:
    if Path(name).exists():
        print(f"Downloading {name} ...")
        files.download(name)
    else:
        print(f"Skip (not found): {name}")

---

### Using local code instead of GitHub

If your changes are not pushed yet, upload files in the cell below (`train_touch.py`, `features.py`, `config_touch.py`, `convert_touch.py`, and optionally `touch_data.jsonl.gz`).

In [ ]:
# Uncomment to upload local files into the current directory:
# from google.colab import files
# uploaded = files.upload()
# print("Uploaded:", list(uploaded.keys()))